# 06 章 / 第 1 节 · DPO 理论:Bradley-Terry → DPO loss 推导

## 学习目标
- 复习 RLHF 三段式(SFT → RM → PPO)和 Bradley-Terry 偏好模型
- 推导 DPO loss:**从 RLHF 最优解的封闭式开始,反代回偏好模型**
- 用 PyTorch 数值验证推导每一步
- 画 chosen vs rejected 的 implicit reward margin 直观图

## 对应八股
`llm-interview/llm-rlhf.md`(DPO 节)

## 完全离线,纯 PyTorch,CPU 几秒钟跑通。


\
## 1. 背景:RLHF 三段式

```
SFT-init policy π_SFT
   ↓ 标注偏好 (chosen, rejected)
训 Reward Model r_φ(x, y)
   ↓ 拿 RM 当奖励
PPO 更新 π_θ,目标:max E[r] - β · KL(π_θ ‖ π_SFT)
```

**KL 项的意义**:防止 π_θ 跑偏太远(reward hacking)。

## 2. RLHF 的最优解

带 KL 约束的策略优化目标:
$$
\max_\pi \; \mathbb{E}_{x, y \sim \pi(\cdot|x)} [r(x, y)] - \beta \, \text{KL}\big(\pi(\cdot|x) \,\|\, \pi_{\text{ref}}(\cdot|x)\big)
$$

这个目标有**闭式解**(Rafailov et al. 2023 / DPO 论文附录 A.1):
$$
\pi^*(y | x) = \frac{1}{Z(x)} \, \pi_{\text{ref}}(y | x) \, \exp\!\left(\frac{1}{\beta} \, r(x, y)\right)
$$

其中 `Z(x)` 是归一化项(partition function)。

## 3. DPO 的关键洞察

**把上式反过来解 `r(x, y)`**:
$$
r(x, y) = \beta \log \frac{\pi^*(y | x)}{\pi_{\text{ref}}(y | x)} + \beta \log Z(x)
$$

也就是说:**reward 可以用 policy 与 ref policy 的 log-ratio 表达**,
还多了一个只依赖 `x` 的常数 `β log Z(x)`。

## 4. 代回 Bradley-Terry

Bradley-Terry 偏好模型:
$$
P(y_w \succ y_l \mid x) = \sigma\!\big(r(x, y_w) - r(x, y_l)\big)
$$

把 step 3 的 `r` 代入,**`β log Z(x)` 这一项在差里消掉**:
$$
P(y_w \succ y_l) = \sigma\!\left(\beta \log \frac{\pi^*(y_w | x)}{\pi_{\text{ref}}(y_w | x)} - \beta \log \frac{\pi^*(y_l | x)}{\pi_{\text{ref}}(y_l | x)}\right)
$$

DPO loss 就是这个概率的负对数似然(对偏好数据集求均值):
$$
\mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = -\mathbb{E}_{(x, y_w, y_l)} \log \sigma\!\left(\beta \log \frac{\pi_\theta(y_w | x)}{\pi_{\text{ref}}(y_w | x)} - \beta \log \frac{\pi_\theta(y_l | x)}{\pi_{\text{ref}}(y_l | x)}\right)
$$

**绕开了**:RM 训练、PPO 采样、Z(x) 计算。一个分类 loss 解决战斗。


## 5. 用 PyTorch 实现 DPO loss(只算一遍,看看长什么样)


In [ ]:
import torch
import torch.nn.functional as F


def dpo_loss(
    policy_chosen_logps: torch.Tensor,      # (B,) log π_θ(y_w | x) 整段 sum
    policy_rejected_logps: torch.Tensor,    # (B,) log π_θ(y_l | x)
    ref_chosen_logps: torch.Tensor,         # (B,) log π_ref(y_w | x)
    ref_rejected_logps: torch.Tensor,       # (B,)
    beta: float = 0.1,
):
    """DPO loss 的完整实现,12 行写完。"""
    # 1. policy 与 ref 的 log-ratio
    policy_ratios = policy_chosen_logps - policy_rejected_logps     # (B,)
    ref_ratios    = ref_chosen_logps - ref_rejected_logps           # (B,)
    # 2. 两个 ratio 的差(就是 implicit reward margin)
    logits = beta * (policy_ratios - ref_ratios)
    # 3. 二分类 NLL == -log sigmoid(logits)
    loss = -F.logsigmoid(logits).mean()
    # 4. 监控指标:implicit reward
    chosen_reward   = beta * (policy_chosen_logps   - ref_chosen_logps)
    rejected_reward = beta * (policy_rejected_logps - ref_rejected_logps)
    return loss, chosen_reward, rejected_reward


# 验证:模拟一批样本
torch.manual_seed(0)
B = 4
# 假设 policy 已经学得有点偏向 chosen(chosen logp 比 rejected 高一些)
policy_chosen   = torch.tensor([-10.2, -8.5, -15.1, -9.8])
policy_rejected = torch.tensor([-12.5, -10.0, -16.0, -11.2])
ref_chosen      = torch.tensor([-11.0, -9.5, -15.8, -10.5])
ref_rejected    = torch.tensor([-11.5, -9.0, -15.5, -10.3])

loss, cr, rr = dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=0.1)
print(f"DPO loss:       {loss.item():.4f}")
print(f"chosen reward:   {cr.tolist()}")
print(f"rejected reward: {rr.tolist()}")
print(f"reward margin:   {(cr - rr).tolist()}  ← 正 = policy 偏向 chosen,好")


## 6. 画 implicit reward margin 随训练步数变化(模拟)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 模拟 200 步训练:policy 逐步学着把 chosen logp 拉高
np.random.seed(0)
steps = np.arange(200)
# 初始 chosen ≈ rejected,逐步分开
gap = 0.01 * steps + np.random.randn(200) * 0.3
chosen_r   = gap / 2
rejected_r = -gap / 2
margin = chosen_r - rejected_r

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(steps, chosen_r, label="chosen", color="C2")
ax[0].plot(steps, rejected_r, label="rejected", color="C3")
ax[0].set_title("implicit reward (policy vs ref log-ratio)")
ax[0].set_xlabel("step"); ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(steps, margin, color="C0")
ax[1].axhline(0, color="gray", linestyle="--", alpha=0.5)
ax[1].set_title("reward margin (chosen - rejected)  应单调上升")
ax[1].set_xlabel("step"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("健康训练:margin 单调上升,但不要追到天上 —— 太高说明 policy 偏离 ref 太远,质量在退化")


## 7. 关键性质(面试必考)

### Q1:为什么 DPO 训练不需要 sample?

A:`L_DPO` 只用 (chosen, rejected) 这对**已有**数据计算 logp,
不像 PPO 要 on-policy 采样 → 算 advantage → 更新。

### Q2:`β` 控制什么?

A:KL 约束强度。
- `β` 大(0.5+):policy 不敢偏离 ref → reward margin 涨不上去
- `β` 小(0.01-):policy 自由发挥 → margin 涨快但**生成质量退化**
- 经验起步 `β=0.1`,生成评测下降就回调到 0.3

### Q3:Z(x) 跑哪去了?

A:`Z(x)` 只依赖 `x`,在偏好概率的差里消掉了:
`P(y_w ≻ y_l) = σ(r(y_w) - r(y_l))`,两个 `β log Z(x)` 一减为 0。
这是 DPO 能绕开 RM 的**数学根源**。

### Q4:DPO 有什么缺点?

A:
1. 对偏好数据质量极敏感(没有 RM 这一层平滑)
2. 退化时无法察觉(没有 RM 监控信号),要靠生成评测
3. 不能利用 unlabeled prompts(PPO 可以)

## 8. 延伸阅读

- 论文:`/paper fetch 2305.18290` (DPO 原论文)
- 论文:`/paper fetch 2402.01306` (KTO,DPO 的简化版)
- 论文:`/paper fetch 2405.14734` (SimPO,无 ref policy 版 DPO)
- 本仓:[`02_dpo_unsloth.ipynb`](02_dpo_unsloth.ipynb)(真训一遍)
- 本仓:[`03_grpo_verifiable_reward.ipynb`](03_grpo_verifiable_reward.ipynb)(对照组:GRPO)
- [[Slipbox/dpo-derivation]] / [[Slipbox/dpo-vs-ppo-vs-rlhf]]
